<a href="https://colab.research.google.com/github/IreneAbbey/The-Logistics-Auditor/blob/dev/Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# **Story 1: The Schema Builder**

In [5]:
#Load datasets
orders = pd.read_csv('olist_orders_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

In [6]:
master_ds = orders.merge(reviews, on='order_id', how='inner') \
                  .merge(customers, on='customer_id', how='inner')

In [8]:
#Ensure no duplicate rows were introduced by the joins
print("Duplicate rows:", master_ds.duplicated().sum())

Duplicate rows: 0


# **Story 2: The "Real" Delay Calculator**

In [43]:
#Convert columns to actual datetime objects
master_ds['order_estimated_delivery_date'] = pd.to_datetime(master_ds['order_estimated_delivery_date'])
master_ds['order_delivered_customer_date'] = pd.to_datetime(master_ds['order_delivered_customer_date'])

In [44]:
master_ds['Days_Difference'] = (
    master_ds['order_estimated_delivery_date'] - master_ds['order_delivered_customer_date']
).dt.days

In [71]:
def classify_with_flags(row):
    status = row['order_status']

    if status in ['canceled', 'unavailable']:
        return "Not Delivered"

    if pd.isna(row['order_delivered_customer_date']):
        return "In Progress / Unknown"

    days = row['Days_Difference']

    if days >= 0:
        return "On Time"
    elif days >= -5:
        return "Late"
    else:
        return "Super Late"

master_ds['Delivery_Status'] = master_ds.apply(classify_with_flags, axis=1)